# Feature Count Optimization & Model Training

This notebook:
1. Loads feature schema from `selected_features.json` (from feature selection notebook)
2. Pulls data for ALL seasons (2009-10 through 2023-24)
3. Tests different feature counts to find the optimal set
4. Evaluates using calibration metrics (Brier score), not just accuracy
5. Uses time-based splits to avoid data leakage

**Training Split:**
- Train: 2010-2019 (pre-COVID)
- Validation: 2021-2022 (return to normalcy)
- Test: 2022-2024 (modern NBA, held out)

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import accuracy_score, brier_score_loss
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

print("✅ Imports loaded")

In [ ]:
# Database connection - update with your credentials
import os

from dotenv import load_dotenv

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM team_game_stats"))
    print(f"✅ Connected! team_game_stats has {result.scalar():,} rows")

In [ ]:
# Load feature names from feature selection notebook
with open("selected_features.json") as f:
    SELECTED_FEATURES = json.load(f)

print(f"📋 Loaded {len(SELECTED_FEATURES)} feature names from schema")
print(f"\nSample features: {SELECTED_FEATURES[:5]}")

## 1. Configuration

In [ ]:
# Seasons to pull (NBA season_id format: '2' + year of season start)
# e.g., '22023' = 2023-24 season, '22009' = 2009-10 season

ALL_SEASONS = [
    "22009",
    "22010",
    "22011",
    "22012",
    "22013",  # 2009-10 through 2013-14
    "22014",
    "22015",
    "22016",
    "22017",
    "22018",  # 2014-15 through 2018-19
    "22019",
    "22020",
    "22021",
    "22022",
    "22023",  # 2019-20 through 2023-24
]

# Split by season
TRAIN_SEASONS = [
    "22009",
    "22010",
    "22011",
    "22012",
    "22013",
    "22014",
    "22015",
    "22016",
    "22017",
    "22018",
]
VAL_SEASONS = ["22022"]  # Post-COVID return
TEST_SEASONS = ["22023"]  # Modern NBA holdout

# Note: We skip '22019' (COVID-disrupted bubble season)

print(f"Train seasons: {len(TRAIN_SEASONS)} ({TRAIN_SEASONS[0]} to {TRAIN_SEASONS[-1]})")
print(f"Val seasons: {len(VAL_SEASONS)} ({VAL_SEASONS})")
print(f"Test seasons: {len(TEST_SEASONS)} ({TEST_SEASONS})")

## 2. Data Loading Functions

Same queries as feature selection notebook, but parameterized for season list.

In [ ]:
def get_team_data(engine, seasons):
    """
    Pull team-level rolling average stats for specified seasons.
    Returns one row per game with home team stats, away team stats, and differentials.
    """
    season_list = ", ".join([f"'{s}'" for s in seasons])

    query = text(f"""
        WITH game_teams AS (
            SELECT DISTINCT
                tgs.game_id,
                tgs.season_id,
                tgs.team_game_date as game_date,
                MAX(CASE WHEN tgs.team_matchup LIKE '%vs.%' THEN tgs.team_id END) as team1_id,
                MAX(CASE WHEN tgs.team_matchup LIKE '%@%' THEN tgs.team_id END) as team2_id
            FROM team_game_stats tgs
            WHERE tgs.season_id IN ({season_list})
            GROUP BY tgs.game_id, tgs.season_id, tgs.team_game_date
        )
        SELECT 
            gt.game_id,
            gt.season_id,
            gt.game_date,
            gt.team1_id,
            gt.team2_id,
            
            -- Team 1 (HOME) stats
            t1.avg_team_pts as team_pts,
            t1.avg_team_fgm as team_fgm,
            t1.avg_team_fga as team_fga,
            t1.avg_team_fg_pct as team_fg_pct,
            t1.avg_team_fg3m as team_fg3m,
            t1.avg_team_fg3a as team_fg3a,
            t1.avg_team_fg3_pct as team_fg3_pct,
            t1.avg_team_ftm as team_ftm,
            t1.avg_team_fta as team_fta,
            t1.avg_team_ft_pct as team_ft_pct,
            t1.avg_team_oreb as team_oreb,
            t1.avg_team_dreb as team_dreb,
            t1.avg_team_reb as team_reb,
            t1.avg_team_ast as team_ast,
            t1.avg_team_stl as team_stl,
            t1.avg_team_blk as team_blk,
            t1.avg_team_tov as team_tov,
            t1.avg_team_pf as team_pf,
            t1.games_in_average as team_games_in_avg,
            
            -- Team 2 (AWAY) stats
            t2.avg_team_pts as opp_pts,
            t2.avg_team_fgm as opp_fgm,
            t2.avg_team_fga as opp_fga,
            t2.avg_team_fg_pct as opp_fg_pct,
            t2.avg_team_fg3m as opp_fg3m,
            t2.avg_team_fg3a as opp_fg3a,
            t2.avg_team_fg3_pct as opp_fg3_pct,
            t2.avg_team_ftm as opp_ftm,
            t2.avg_team_fta as opp_fta,
            t2.avg_team_ft_pct as opp_ft_pct,
            t2.avg_team_oreb as opp_oreb,
            t2.avg_team_dreb as opp_dreb,
            t2.avg_team_reb as opp_reb,
            t2.avg_team_ast as opp_ast,
            t2.avg_team_stl as opp_stl,
            t2.avg_team_blk as opp_blk,
            t2.avg_team_tov as opp_tov,
            t2.avg_team_pf as opp_pf,
            t2.games_in_average as opp_games_in_avg,
            
            -- Differentials (HOME - AWAY)
            t1.avg_team_pts - t2.avg_team_pts as diff_pts,
            t1.avg_team_fgm - t2.avg_team_fgm as diff_fgm,
            t1.avg_team_fga - t2.avg_team_fga as diff_fga,
            t1.avg_team_fg_pct - t2.avg_team_fg_pct as diff_fg_pct,
            t1.avg_team_fg3m - t2.avg_team_fg3m as diff_fg3m,
            t1.avg_team_fg3a - t2.avg_team_fg3a as diff_fg3a,
            t1.avg_team_fg3_pct - t2.avg_team_fg3_pct as diff_fg3_pct,
            t1.avg_team_ftm - t2.avg_team_ftm as diff_ftm,
            t1.avg_team_fta - t2.avg_team_fta as diff_fta,
            t1.avg_team_ft_pct - t2.avg_team_ft_pct as diff_ft_pct,
            t1.avg_team_oreb - t2.avg_team_oreb as diff_oreb,
            t1.avg_team_dreb - t2.avg_team_dreb as diff_dreb,
            t1.avg_team_reb - t2.avg_team_reb as diff_reb,
            t1.avg_team_ast - t2.avg_team_ast as diff_ast,
            t1.avg_team_stl - t2.avg_team_stl as diff_stl,
            t1.avg_team_blk - t2.avg_team_blk as diff_blk,
            t1.avg_team_tov - t2.avg_team_tov as diff_tov,
            t1.avg_team_pf - t2.avg_team_pf as diff_pf
            
        FROM game_teams gt
        JOIN team_average_game_stats t1 ON gt.game_id = t1.game_id AND gt.team1_id = t1.team_id
        JOIN team_average_game_stats t2 ON gt.game_id = t2.game_id AND gt.team2_id = t2.team_id
        WHERE t1.games_in_average >= 5
          AND t2.games_in_average >= 5
        ORDER BY gt.game_date, gt.game_id;
    """)

    with engine.connect() as conn:
        return pd.read_sql(query, conn)


def get_player_data(engine, seasons):
    """
    Pull player-level rolling average stats for specified seasons.
    Returns top 10 players by minutes for each team in each game.
    """
    season_list = ", ".join([f"'{s}'" for s in seasons])

    query = text(f"""
        WITH game_teams AS (
            SELECT DISTINCT
                tgs.game_id,
                tgs.season_id,
                MAX(CASE WHEN tgs.team_matchup LIKE '%vs.%' THEN tgs.team_id END) as team1_id,
                MAX(CASE WHEN tgs.team_matchup LIKE '%@%' THEN tgs.team_id END) as team2_id
            FROM team_game_stats tgs
            WHERE tgs.season_id IN ({season_list})
            GROUP BY tgs.game_id, tgs.season_id
        ),
        ranked_players AS (
            SELECT 
                pgs.game_id,
                pgs.season_id,
                pgs.team_id,
                pgs.player_id,
                pgs.avg_min,
                pgs.avg_pts,
                pgs.avg_fgm,
                pgs.avg_fga,
                pgs.avg_fg_pct,
                pgs.avg_fg3m,
                pgs.avg_fg3a,
                pgs.avg_fg3_pct,
                pgs.avg_ftm,
                pgs.avg_fta,
                pgs.avg_ft_pct,
                pgs.avg_oreb,
                pgs.avg_dreb,
                pgs.avg_reb,
                pgs.avg_ast,
                pgs.avg_stl,
                pgs.avg_blk,
                pgs.avg_tov,
                pgs.avg_pf,
                pgs.avg_plus_minus,
                pgs.games_in_average,
                ROW_NUMBER() OVER (PARTITION BY pgs.game_id, pgs.team_id ORDER BY pgs.avg_min DESC) as player_rank
            FROM player_average_game_stats pgs
            WHERE pgs.avg_min > 0
        )
        -- HOME team players
        SELECT 
            gt.game_id,
            gt.season_id,
            'team' as player_team_type,
            gt.team1_id as team_id,
            rp.player_rank,
            rp.player_id,
            rp.avg_min,
            rp.avg_pts,
            rp.avg_fgm,
            rp.avg_fga,
            rp.avg_fg_pct,
            rp.avg_fg3m,
            rp.avg_fg3a,
            rp.avg_fg3_pct,
            rp.avg_ftm,
            rp.avg_fta,
            rp.avg_ft_pct,
            rp.avg_oreb,
            rp.avg_dreb,
            rp.avg_reb,
            rp.avg_ast,
            rp.avg_stl,
            rp.avg_blk,
            rp.avg_tov,
            rp.avg_pf,
            rp.avg_plus_minus,
            rp.games_in_average
        FROM game_teams gt
        JOIN ranked_players rp ON gt.game_id = rp.game_id AND gt.team1_id = rp.team_id
        WHERE rp.player_rank <= 10
        
        UNION ALL
        
        -- AWAY team players
        SELECT 
            gt.game_id,
            gt.season_id,
            'opp' as player_team_type,
            gt.team2_id as team_id,
            rp.player_rank,
            rp.player_id,
            rp.avg_min,
            rp.avg_pts,
            rp.avg_fgm,
            rp.avg_fga,
            rp.avg_fg_pct,
            rp.avg_fg3m,
            rp.avg_fg3a,
            rp.avg_fg3_pct,
            rp.avg_ftm,
            rp.avg_fta,
            rp.avg_ft_pct,
            rp.avg_oreb,
            rp.avg_dreb,
            rp.avg_reb,
            rp.avg_ast,
            rp.avg_stl,
            rp.avg_blk,
            rp.avg_tov,
            rp.avg_pf,
            rp.avg_plus_minus,
            rp.games_in_average
        FROM game_teams gt
        JOIN ranked_players rp ON gt.game_id = rp.game_id AND gt.team2_id = rp.team_id
        WHERE rp.player_rank <= 10
        
        ORDER BY game_id, player_team_type, player_rank;
    """)

    with engine.connect() as conn:
        return pd.read_sql(query, conn)

In [ ]:
def flatten_player_data(player_df):
    """
    Pivot player data from long format (multiple rows per game)
    to wide format (one row per game with player stats as columns).
    """
    stat_cols = [
        "avg_min",
        "avg_pts",
        "avg_fgm",
        "avg_fga",
        "avg_fg_pct",
        "avg_fg3m",
        "avg_fg3a",
        "avg_fg3_pct",
        "avg_ftm",
        "avg_fta",
        "avg_ft_pct",
        "avg_oreb",
        "avg_dreb",
        "avg_reb",
        "avg_ast",
        "avg_stl",
        "avg_blk",
        "avg_tov",
        "avg_pf",
        "avg_plus_minus",
        "games_in_average",
    ]

    # Player counts per team per game
    player_counts = player_df.groupby(["game_id", "player_team_type"]).size().unstack(fill_value=0)
    player_counts.columns = ["opp_player_count", "team_player_count"]
    player_counts = player_counts.reset_index()

    # Create column prefix
    player_df = player_df.copy()
    player_df["player_prefix"] = (
        player_df["player_team_type"] + "_player" + player_df["player_rank"].astype(str)
    )

    # Pivot each stat
    pivoted_parts = []
    for stat in stat_cols:
        stat_pivot = player_df.pivot_table(
            index="game_id", columns="player_prefix", values=stat, aggfunc="first"
        )
        stat_pivot.columns = [f"{col}_{stat}" for col in stat_pivot.columns]
        pivoted_parts.append(stat_pivot)

    flattened = pd.concat(pivoted_parts, axis=1).reset_index()
    flattened = flattened.merge(player_counts, on="game_id", how="left")
    flattened["diff_player_count"] = flattened["team_player_count"] - flattened["opp_player_count"]

    # Fill missing player slots with 0
    nan_before = flattened.isna().sum().sum()
    flattened = flattened.fillna(0)
    print(f"   Filled {nan_before} NaN values with 0")

    return flattened


def add_target_variable(ml_df, engine, seasons):
    """
    Add binary target: 1 if home team (team1) won, 0 otherwise.
    """
    season_list = ", ".join([f"'{s}'" for s in seasons])

    query = text(f"""
        SELECT game_id, team_id, team_pts
        FROM team_game_stats
        WHERE season_id IN ({season_list})
    """)

    with engine.connect() as conn:
        results_df = pd.read_sql(query, conn)

    team1_results = results_df.rename(
        columns={"team_id": "team1_id", "team_pts": "team1_actual_pts"}
    )
    team2_results = results_df.rename(
        columns={"team_id": "team2_id", "team_pts": "team2_actual_pts"}
    )

    ml_df = ml_df.merge(
        team1_results[["game_id", "team1_id", "team1_actual_pts"]],
        on=["game_id", "team1_id"],
        how="left",
    )
    ml_df = ml_df.merge(
        team2_results[["game_id", "team2_id", "team2_actual_pts"]],
        on=["game_id", "team2_id"],
        how="left",
    )

    ml_df["target_win"] = (ml_df["team1_actual_pts"] > ml_df["team2_actual_pts"]).astype(int)
    ml_df = ml_df.drop(columns=["team1_actual_pts", "team2_actual_pts"])

    return ml_df

## 3. Load Data for All Seasons

In [ ]:
# Pull data for all seasons (this may take a few minutes)
print("⏳ Loading team data...")
team_df = get_team_data(engine, ALL_SEASONS)
print(f"   Team data: {len(team_df):,} games")

print("\n⏳ Loading player data...")
player_df = get_player_data(engine, ALL_SEASONS)
print(f"   Player data: {len(player_df):,} player-game rows")

print("\n⏳ Flattening player data...")
player_flat = flatten_player_data(player_df)
print(f"   Flattened: {len(player_flat):,} games, {len(player_flat.columns)} columns")

In [ ]:
# Merge team and player data
print("⏳ Merging team and player data...")
ml_df = team_df.merge(player_flat, on="game_id", how="inner")
print(f"   Merged: {len(ml_df):,} games")

# Add target variable
print("\n⏳ Adding target variable...")
ml_df = add_target_variable(ml_df, engine, ALL_SEASONS)
print(f"   Target distribution: {ml_df['target_win'].mean():.1%} home wins")

# Sort by date (critical for time-based split!)
ml_df = ml_df.sort_values("game_date").reset_index(drop=True)
print(f"\n✅ Final dataset: {len(ml_df):,} games, {len(ml_df.columns)} columns")
print(f"   Date range: {ml_df['game_date'].min()} to {ml_df['game_date'].max()}")

In [ ]:
# Filter to only selected features (from feature selection notebook)
LEAKAGE_COLS = [
    "game_id",
    "season_id",
    "game_date",
    "team1_id",
    "team2_id",
    "team_games_in_avg",
    "opp_games_in_avg",
]

# Keep: selected features + target + metadata for splitting
available_features = [f for f in SELECTED_FEATURES if f in ml_df.columns]
missing_features = [f for f in SELECTED_FEATURES if f not in ml_df.columns]

print(f"Features from schema: {len(SELECTED_FEATURES)}")
print(f"Available in data: {len(available_features)}")
if missing_features:
    print(f"⚠️ Missing features: {missing_features[:10]}...")

# Create filtered dataframe with metadata for splitting
keep_cols = ["game_id", "season_id", "game_date"] + available_features + ["target_win"]
filtered_df = ml_df[keep_cols].copy()

print(f"\n✅ Filtered dataset: {len(filtered_df):,} games, {len(available_features)} features")

## 4. Create Train/Val/Test Splits by Season

In [ ]:
# Split by season
train_df = filtered_df[filtered_df["season_id"].isin(TRAIN_SEASONS)].copy()
val_df = filtered_df[filtered_df["season_id"].isin(VAL_SEASONS)].copy()
test_df = filtered_df[filtered_df["season_id"].isin(TEST_SEASONS)].copy()

print(f"Train: {len(train_df):,} games ({TRAIN_SEASONS[0]} to {TRAIN_SEASONS[-1]})")
print(f"Val:   {len(val_df):,} games ({VAL_SEASONS})")
print(f"Test:  {len(test_df):,} games ({TEST_SEASONS})")

# Prepare X and y
feature_cols = available_features

X_train = train_df[feature_cols]
y_train = train_df["target_win"]

X_val = val_df[feature_cols]
y_val = val_df["target_win"]

X_test = test_df[feature_cols]
y_test = test_df["target_win"]

print(f"\nFeatures: {len(feature_cols)}")
print(f"Train home win rate: {y_train.mean():.1%}")
print(f"Val home win rate: {y_val.mean():.1%}")
print(f"Test home win rate: {y_test.mean():.1%}")

## 5. Compute Feature Importance Scores (on Training Data Only)

In [ ]:
# IMPORTANT: Compute scores on TRAINING data only to avoid leakage
print("⏳ Computing ANOVA scores on training data...")
f_scores, p_vals = f_classif(X_train, y_train)
anova_df = (
    pd.DataFrame({"feature": X_train.columns, "f_score": f_scores, "p_value": p_vals})
    .sort_values("f_score", ascending=False)
    .reset_index(drop=True)
)

print("⏳ Computing Mutual Information scores...")
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = (
    pd.DataFrame({"feature": X_train.columns, "mi_score": mi_scores})
    .sort_values("mi_score", ascending=False)
    .reset_index(drop=True)
)

print("\n✅ Scoring complete!")
print("\n🏆 Top 15 by ANOVA:")
print(anova_df.head(15).to_string(index=False))
print("\n🏆 Top 15 by Mutual Information:")
print(mi_df.head(15).to_string(index=False))

## 6. Feature Count Optimization

In [ ]:
def select_top_features(anova_df, mi_df, n_features, method="union"):
    """Select top N features using ANOVA, MI, or union of both."""
    if method == "anova":
        return anova_df.head(n_features)["feature"].tolist()
    elif method == "mi":
        return mi_df.head(n_features)["feature"].tolist()
    elif method == "union":
        anova_top = set(anova_df.head(n_features)["feature"])
        mi_top = set(mi_df.head(n_features)["feature"])
        combined = list(anova_top | mi_top)
        if len(combined) > n_features:
            anova_ranks = {f: i for i, f in enumerate(anova_df["feature"])}
            mi_ranks = {f: i for i, f in enumerate(mi_df["feature"])}
            avg_ranks = {f: (anova_ranks.get(f, 999) + mi_ranks.get(f, 999)) / 2 for f in combined}
            combined = sorted(combined, key=lambda f: avg_ranks[f])[:n_features]
        return combined


def evaluate_feature_counts(X_train, y_train, X_val, y_val, anova_df, mi_df, feature_counts):
    """Test different feature counts on validation set."""
    results = []

    for n in feature_counts:
        selected = select_top_features(anova_df, mi_df, n, method="union")
        selected = [f for f in selected if f in X_train.columns]

        if len(selected) < 5:
            continue

        X_tr = X_train[selected]
        X_va = X_val[selected]

        # Train XGBoost
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            reg_lambda=1.0,
            reg_alpha=0.1,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0,
        )
        model.fit(X_tr, y_train)

        # Raw scores
        raw_probs = model.predict_proba(X_va)[:, 1]
        raw_brier = brier_score_loss(y_val, raw_probs)
        raw_acc = accuracy_score(y_val, (raw_probs >= 0.5).astype(int))

        # Calibrated scores
        calibrated = CalibratedClassifierCV(model, method="isotonic", cv="prefit")
        calibrated.fit(X_tr, y_train)
        cal_probs = calibrated.predict_proba(X_va)[:, 1]
        cal_brier = brier_score_loss(y_val, cal_probs)
        cal_acc = accuracy_score(y_val, (cal_probs >= 0.5).astype(int))

        results.append(
            {
                "n_features": n,
                "actual_features": len(selected),
                "raw_brier": raw_brier,
                "raw_accuracy": raw_acc,
                "cal_brier": cal_brier,
                "cal_accuracy": cal_acc,
                "features": selected,
            }
        )

        print(
            f"n={n:3d} ({len(selected):3d}) | Raw: Brier={raw_brier:.4f} Acc={raw_acc:.3f} | Cal: Brier={cal_brier:.4f} Acc={cal_acc:.3f}"
        )

    return results

In [ ]:
FEATURE_COUNTS = [10, 25, 50, 75, 100, 125, 150, 200, 250, 300]

print("🔬 FEATURE COUNT OPTIMIZATION")
print(f"Train: {len(X_train):,} | Val: {len(X_val):,}")
print("=" * 80)

optimization_results = evaluate_feature_counts(
    X_train, y_train, X_val, y_val, anova_df, mi_df, FEATURE_COUNTS
)

In [ ]:
# Find optimal
results_df = pd.DataFrame(optimization_results)
best_idx = results_df["cal_brier"].idxmin()
best = optimization_results[best_idx]

print("\n📊 RESULTS SUMMARY:")
print(
    results_df[
        ["n_features", "actual_features", "raw_brier", "cal_brier", "cal_accuracy"]
    ].to_string(index=False)
)
print(
    f"\n🏆 OPTIMAL: {best['actual_features']} features, Brier={best['cal_brier']:.4f}, Acc={best['cal_accuracy']:.3f}"
)

optimal_features = best["features"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    results_df["actual_features"], results_df["raw_brier"], "o-", label="Raw", color="coral"
)
axes[0].plot(
    results_df["actual_features"],
    results_df["cal_brier"],
    "s-",
    label="Calibrated",
    color="steelblue",
)
axes[0].axvline(best["actual_features"], color="green", linestyle="--", alpha=0.7)
axes[0].set_xlabel("Number of Features")
axes[0].set_ylabel("Brier Score (lower is better)")
axes[0].set_title("Brier Score vs Feature Count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    results_df["actual_features"], results_df["raw_accuracy"], "o-", label="Raw", color="coral"
)
axes[1].plot(
    results_df["actual_features"],
    results_df["cal_accuracy"],
    "s-",
    label="Calibrated",
    color="steelblue",
)
axes[1].axvline(best["actual_features"], color="green", linestyle="--", alpha=0.7)
axes[1].set_xlabel("Number of Features")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs Feature Count")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Final Model: Train on Train+Val, Evaluate on Test

In [ ]:
# Combine train + val for final training
X_trainval = pd.concat([X_train, X_val])[optimal_features]
y_trainval = pd.concat([y_train, y_val])

X_test_final = X_test[optimal_features]

print(f"Final training: {len(X_trainval):,} games")
print(f"Test holdout: {len(X_test_final):,} games")

# Train
final_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    reg_lambda=1.0,
    reg_alpha=0.1,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
final_model.fit(X_trainval, y_trainval)

# Calibrate
final_calibrated = CalibratedClassifierCV(final_model, method="isotonic", cv="prefit")
final_calibrated.fit(X_trainval, y_trainval)

# Evaluate on TEST set
test_probs = final_calibrated.predict_proba(X_test_final)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)
test_brier = brier_score_loss(y_test, test_probs)
test_acc = accuracy_score(y_test, test_preds)

print(f"\n🎯 FINAL TEST RESULTS ({TEST_SEASONS}):")
print(f"   Brier Score: {test_brier:.4f}")
print(f"   Accuracy: {test_acc:.3f}")
print(f"   Baseline: {max(y_test.mean(), 1 - y_test.mean()):.3f}")

In [ ]:
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(8, 8))
CalibrationDisplay.from_predictions(y_test, test_probs, n_bins=10, ax=ax, name="XGBoost + Isotonic")
ax.set_title(f"Calibration Plot - Test Set ({TEST_SEASONS[0]}-{TEST_SEASONS[-1]})")
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Save model
joblib.dump(final_calibrated, "nba_prediction_model.joblib")
print("✅ Saved model to 'nba_prediction_model.joblib'")

# Save optimal features
with open("optimal_features.json", "w") as f:
    json.dump(optimal_features, f, indent=2)
print(f"✅ Saved {len(optimal_features)} features to 'optimal_features.json'")

# Save results
results_df.to_csv("optimization_results.csv", index=False)
print("✅ Saved optimization results to 'optimization_results.csv'")